# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We display the available record sets and their fields using the `@id`.

In [ ]:
# List all record sets and their fields columns by @id
from collections.abc import Iterable
def flatten(x):
    if isinstance(x, Iterable) and not isinstance(x, (str, bytes)):
        for item in x:
            yield from flatten(item)
    elif x is not None:
        yield x

record_sets = getattr(metadata, 'record_set', []) if hasattr(metadata, 'record_set') else []

if not record_sets:
    print("This dataset does not define any explicit Croissant Record Sets at the top-level. Attempting to inspect files in `dataset.files`...")
    # The mlcroissant library will infer file and table based record sets
    if hasattr(dataset, 'files'):
        for f in dataset.files:
            print(f"FileObject @id: {f.id if hasattr(f, 'id') else getattr(f, '@id', None)}  |  name: {f.name if hasattr(f, 'name') else getattr(f, 'name', None)}")
            if hasattr(f, 'record_sets') and f.record_sets:
                for r in f.record_sets:
                    print(f"  Inferred RecordSet @id: {r.id}")
                    if hasattr(r, 'fields'):
                        for fld in r.fields:
                            print(f"    Field @id: {fld.id}, name: {fld.name}")
else:
    for rs in flatten(record_sets):
        print(f"RecordSet @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
        # Print corresponding fields (columns)
        if hasattr(rs, 'fields'):
            for fld in rs.fields:
                print(f"  Field @id: {fld.id}, name: {fld.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Most Croissant datasets infer a single RecordSet containing the tabular data. For this dataset, we'll enumerate available record sets and use their `@id` in code.

In [ ]:
# Collect all record set @id(s) for extraction
# You may manually specify record set ids if not listed in metadata
inferred_record_set_ids = []
if hasattr(metadata, 'record_set'):
    rs = metadata.record_set
    if isinstance(rs, list):
        for r in rs:
            inferred_record_set_ids.append(r.id)
    else:
        inferred_record_set_ids.append(rs.id)
if not inferred_record_set_ids and hasattr(dataset, 'files'):
    for f in dataset.files:
        if hasattr(f, 'record_sets') and f.record_sets:
            for rs in f.record_sets:
                inferred_record_set_ids.append(rs.id)
# For this dataset, the main table record set is likely
if inferred_record_set_ids:
    record_sets = inferred_record_set_ids
else:
    raise Exception('No Record Sets found in automatic scan.')

# Load each RecordSet as a DataFrame
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    # Print first few columns and sample
    print(f'Columns for RecordSet @id: {record_set_id}')
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes to prepare for analysis.

We select a numeric field by its `@id` (see the output and adjust as needed).

In [ ]:
# Choose the first inferred record set for demonstration
record_set_id = record_sets[0]
df = dataframes[record_set_id]

print('Available columns:')
print(df.columns.tolist())

# Choose a numeric field, e.g., 'cr:field:MW' for molecular weight, common in proteomics
numeric_field = None
candidate_cols = [col for col in df.columns if ('weight' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower() or 'count' in col.lower() or 'cr:field:MW' == col)]
if candidate_cols:
    numeric_field = candidate_cols[0]
else:
    numeric_field = df.columns[0] # Fallback
print(f"Using numeric field for analysis: {numeric_field}")

# Remove rows where the numeric field is missing
df_filtered = df.copy()
df_filtered = df_filtered[df_filtered[numeric_field].notnull()]

# Try to cast the numeric column to float
try:
    df_filtered[numeric_field] = df_filtered[numeric_field].astype(float)
except Exception as e:
    print(f"Warning: Could not cast {numeric_field} to float. Data type is {df_filtered[numeric_field].dtype}.")

threshold = df_filtered[numeric_field].mean() if df_filtered[numeric_field].dtype.kind in 'fi' else 0

filtered_df = df_filtered[df_filtered[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g. accession, cr:field:accession, or 'cr:field:description')
group_field = None
cat_candidates = [col for col in df.columns if 'accession' in col.lower() or 'description' in col.lower() or col.startswith('cr:field:') or df[col].dtype==object]
if cat_candidates:
    group_field = cat_candidates[0]
if group_field:
    print(f"Grouping by field {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print("Grouped data sample:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df_filtered[numeric_field].dropna(), bins=30, kde=True, color='steelblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If a second numeric field exists, show a scatterplot
possible_numeric = [col for col in df.columns if col != numeric_field and df[col].dtype.kind in 'fi']
if possible_numeric:
    plt.figure(figsize=(6,6))
    plt.scatter(df_filtered[numeric_field], df_filtered[possible_numeric[0]], alpha=0.4)
    plt.xlabel(numeric_field)
    plt.ylabel(possible_numeric[0])
    plt.title(f'Scatter: {numeric_field} vs. {possible_numeric[0]}')
    plt.show()

## 6. Conclusion
This notebook provided a programmatic approach to exploring the FAIR^2 mass spectrometry protein dataset using the Croissant schema and the `mlcroissant` library. Key steps included loading the data, identifying available record sets and fields by their `@id`, filtering and transforming numeric columns, and visualizing data distributions.

With this approach, you can efficiently extract and analyze complex tabular datasets in accordance with FAIR data principles, using only the Croissant schema and Python code.